# Blindspot · GRPO Training (Colab-ready)

Trains **Qwen3.5-9B (4-bit)** with TRL + Unsloth + GRPO against the live Blindspot env. Designed for a single A100 (80 GB) on Colab Pro+ or a HuggingFace Compute credit.

**Stages**
1. Install `unsloth`, `trl`, `openenv-core`, `vllm`
2. Clone the env repo and boot the FastAPI server in the background
3. Load Qwen3.5-9B (4-bit), attach SFT LoRA adapter
4. Build the prompt dataset (one row per fresh `reset`)
5. Define the reward function (HTTP rollout against the env)
6. `GRPOTrainer.train()`
7. Push adapter to the HF Hub + emit reward/loss plots into `plots/`

In [ ]:
%%bash
pip install -q --upgrade unsloth trl 'openenv-core[core]' vllm matplotlib datasets peft accelerate bitsandbytes
pip install -q --upgrade requests websocket-client

In [ ]:
%%bash
git clone https://github.com/vasarlalikhilavinash/blindspot-env || true
cd blindspot-env && python scripts/build_synthetic_seed.py
(cd blindspot-env && uvicorn server.app:app --host 0.0.0.0 --port 8000 &)
sleep 5 && curl -s http://localhost:8000/state | head -c 200

## Load model + SFT adapter

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen3.5-9B-bnb-4bit',
    max_seq_length=4096+128,
    load_in_4bit=True,
)
# Optional: attach SFT adapter trained locally on the M5 Pro
import os
if os.path.isdir('blindspot-env/training/checkpoints/sft'):
    model.load_adapter('blindspot-env/training/checkpoints/sft')
FastLanguageModel.for_training(model)

## Build prompts + reward function (matches `training/grpo_train.py`)

In [ ]:
import sys; sys.path.insert(0, 'blindspot-env')
from training.grpo_train import SYSTEM_PROMPT, render_obs, parse_action
import requests, json, random
from datasets import Dataset

ENV_URL = 'http://localhost:8000'
r0 = requests.post(f'{ENV_URL}/reset', json={}).json()
user_pool = (r0.get('observation', r0) or {}).get('user_id_pool', [])
rng = random.Random(0)
rows = []
for _ in range(256):
    uid = rng.choice(user_pool); seed = rng.randrange(1_000_000)
    obs = requests.post(f'{ENV_URL}/reset', json={'user_id': uid, 'seed': seed}).json()
    obs = obs.get('observation', obs)
    rows.append({'prompt': [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': render_obs(obs)},
    ], 'user_id': uid, 'seed': seed})
ds = Dataset.from_list(rows)
print(f'{len(ds)} prompts')


In [ ]:
def reward_fn(prompts, completions, user_id=None, seed=None, **kw):
    out = []
    uids  = user_id if isinstance(user_id, list) else [user_id]*len(completions)
    seeds = seed    if isinstance(seed, list)    else [seed]*len(completions)
    for prompt_msgs, completion, uid, sd in zip(prompts, completions, uids, seeds):
        text = completion if isinstance(completion, str) else completion[-1].get('content', '')
        action = parse_action(text)
        if not action or 'type' not in action:
            out.append(-0.05); continue
        try:
            payload = {}
            if uid is not None: payload['user_id'] = uid
            if sd  is not None: payload['seed']    = sd
            requests.post(f'{ENV_URL}/reset', json=payload).raise_for_status()
            r = requests.post(f'{ENV_URL}/step', json={'action': action}).json()
            rs = requests.post(f'{ENV_URL}/step', json={'action': {'type': 'stop'}}).json()
            br = (rs.get('observation', rs) or {}).get('reward_breakdown') or {}
            out.append(float(br.get('total', r.get('reward', 0.0) or 0.0)))
        except Exception:
            out.append(-0.1)
    return out


In [ ]:
from trl import GRPOConfig, GRPOTrainer
cfg = GRPOConfig(
    output_dir='blindspot-env/training/checkpoints/grpo',
    learning_rate=5e-6, max_steps=400, num_generations=8,
    per_device_train_batch_size=1, gradient_accumulation_steps=4,
    max_prompt_length=4096, max_completion_length=64,
    logging_steps=5, save_steps=100, bf16=True, report_to='none',
)
trainer = GRPOTrainer(model=model, processing_class=tokenizer,
                       reward_funcs=[reward_fn], args=cfg, train_dataset=ds)
trainer.train()
trainer.save_model('blindspot-env/training/checkpoints/grpo')

## Plot reward + loss curves

In [ ]:
!cd blindspot-env && python scripts/make_plots.py
from IPython.display import Image
Image('blindspot-env/plots/learning_curve.png')

## Push adapter to the Hub

In [ ]:
# from huggingface_hub import login; login()
# model.push_to_hub('your-org/blindspot-qwen35-9b-grpo')